# 🌴 Tropical Diseases — Knowledge Graph Diagnostic Evaluation & Testing
### Interactive Querying, Multi-Mode Benchmarks, Health Inspection & Graph Explorer

This notebook connects directly to your **pre-built Tropical Diseases Knowledge Graph** (indexed from 1,797 clinical case reports) stored in your Google Drive or local storage.

**Key Features:**
- ⚡ **Zero Ingestion Overhead:** No mock LLMs, chunking, or data parsing. Starts in **< 1 minute**.
- 🛡️ **Read-Only Safety:** Connects directly in query mode without altering or overwriting your graph.
- 📊 **Graph Health Diagnostics:** Node & edge census, graph density, and top hub medical entities.
- 🔬 **5-Mode Comparison Benchmark:** Side-by-side evaluation of `naive` (baseline vector RAG) vs `local` vs `global` vs `hybrid` vs `mix`.
- 🩺 **Clinical Benchmark Suite:** Pre-configured diagnostic queries across tropical infectious diseases.
- 🕸️ **Visual Graph Explorer:** 2-hop ego-network visualization and clinical reasoning path finding.
- 💬 **Interactive Doctor Console:** Ask any custom clinical diagnostic question on demand.

| Runtime | Storage Path |
| :--- | :--- |
| **Google Colab (T4 GPU)** | `/content/drive/MyDrive/tropical_kg_storage` (Drive Mount) |
| **Apple Silicon Mac (Local)** | `./tropical_kg_storage` |

---
## Step 1: Environment & Hardware Setup (~1 minute)
Verifies GPU access, starts the background Ollama server, and verifies the required models:
1. **`bge-m3`** — 1024-dim medical embedding model (used for query vector matching).
2. **`qwen2.5:3b`** (or `qwen2.5:7b`) — LLM for synthesizing clinical answers.

In [ ]:
import os, sys, subprocess, time, urllib.request

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running on Google Colab")
    !nvidia-smi
    # Install dependencies
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio networkx matplotlib
    print("✅ Colab packages installed!")
else:
    print("💻 Running locally on Mac / DataSpell")
    try:
        import lightrag, ollama, networkx, matplotlib
        print("✅ Local environment packages verified!")
    except ImportError as e:
        print(f"Installing missing package: {e}")
        !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio networkx matplotlib

# --- Start Ollama server if not already running ---
ollama_bin = "ollama" if IN_COLAB else "/usr/local/bin/ollama"
server_running = False
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as resp:
        if resp.status == 200:
            server_running = True
            print("✅ Ollama server is already running!")
except Exception:
    pass

if not server_running:
    print("Starting Ollama server...")
    os.environ["OLLAMA_NUM_PARALLEL"] = "4"
    subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ)
    time.sleep(3)
    print("✅ Ollama server started!")

# --- Models ---
EMBEDDING_MODEL = "bge-m3"
QUERY_LLM_MODEL = "qwen2.5:3b"  # Change to "qwen2.5:7b" for deeper medical reasoning on Colab T4

print(f"\nVerifying embedding model ({EMBEDDING_MODEL})...")
!ollama pull {EMBEDDING_MODEL}

print(f"\nVerifying query LLM ({QUERY_LLM_MODEL})...")
!ollama pull {QUERY_LLM_MODEL}

print("\n--- Installed Models ---")
!ollama list

---
## Step 2: Connect to Storage & Health Check Diagnostics 🩺
Connects to your Google Drive storage and inspects the internal structures of the Knowledge Graph:
- Verifies all **1,797 clinical cases** are present.
- Inspects `graph_chunk_entity_relation.graphml` to report total nodes and relationship edges.
- Displays the **Top 15 Hub Entities** (the most interconnected medical entities in the graph).

In [ ]:
import os, sys, json
import networkx as nx

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_DIR = "/content/drive/MyDrive/tropical_kg_storage"
else:
    possible_local_paths = [
        "./tropical_kg_storage",
        "../3_LightRAG_Knowledge_Graph/tropical_kg_storage",
        "/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/Custom_Tropical_Project/3_LightRAG_Knowledge_Graph/tropical_kg_storage"
    ]
    STORAGE_DIR = next((p for p in possible_local_paths if os.path.exists(p)), "./tropical_kg_storage")

WORKSPACE = "tropical_kg"
WORKSPACE_DIR = os.path.join(STORAGE_DIR, WORKSPACE)

print("=" * 65)
print("📊 TROPICAL DISEASES KNOWLEDGE GRAPH HEALTH CHECK")
print("=" * 65)
print(f"📁 Storage Directory: {STORAGE_DIR}")
print(f"🏢 Workspace:         {WORKSPACE_DIR}")

if not os.path.exists(WORKSPACE_DIR):
    raise FileNotFoundError(f"❌ Workspace folder not found at: {WORKSPACE_DIR}. Verify Google Drive mount path!")

# 1. Check indexed cases
doc_file = os.path.join(WORKSPACE_DIR, "kv_store_full_docs.json")
if os.path.exists(doc_file):
    with open(doc_file, "r", encoding="utf-8") as f:
        docs = json.load(f)
    print(f"✅ Indexed Patient Cases: {len(docs)} / 1797 cases")
else:
    print("⚠️ kv_store_full_docs.json not found")

# 2. Check GraphML file & load into NetworkX
graph_file = os.path.join(WORKSPACE_DIR, "graph_chunk_entity_relation.graphml")
if os.path.exists(graph_file):
    print(f"Loading GraphML ({os.path.getsize(graph_file) / (1024*1024):.2f} MB)...")
    G = nx.read_graphml(graph_file)
    print(f"✅ Total Clinical Entities (Nodes):     {G.number_of_nodes():,}")
    print(f"✅ Total Diagnostic Relations (Edges): {G.number_of_edges():,}")
    print(f"✅ Graph Density:                       {nx.density(G):.6f}")
    print(f"✅ Connected Components:                {nx.number_connected_components(G)}")

    # 3. Top Hub Entities
    print("\n🏆 Top 15 Most Connected Hub Entities in the Knowledge Graph:")
    print("-" * 65)
    sorted_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:15]
    for rank, (node, deg) in enumerate(sorted_nodes, 1):
        node_data = G.nodes[node]
        entity_type = node_data.get("entity_type", "Unknown")
        print(f"  {rank:2d}. [{entity_type:15s}] {node:<35s} (connections: {deg})")
else:
    print("⚠️ graph_chunk_entity_relation.graphml not found")
print("=" * 65)

---
## Step 3: Initialize LightRAG in Query-Only Mode 🚀
Initializes the `LightRAG` client pointing directly to the verified `tropical_kg` workspace.
- **Read-Only Safety:** Does not run ingestion or extraction pipelines.
- **Ollama Integration:** Uses `bge-m3` for vector search and `qwen2.5:3b` for diagnostic answer synthesis.

In [ ]:
import asyncio
from functools import partial
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc

try:
    import nest_asyncio
except ImportError:
    import nest_asyncio2 as nest_asyncio
nest_asyncio.apply()

EMBED_DIM = 1024

print("Initializing LightRAG Query Engine...")
rag = LightRAG(
    working_dir=STORAGE_DIR,
    workspace=WORKSPACE,
    llm_model_func=ollama_model_complete,
    llm_model_name=QUERY_LLM_MODEL,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=4096,
        func=partial(ollama_embed.func, embed_model=EMBEDDING_MODEL)
    )
)

await rag.initialize_storages()
print(f"✅ LightRAG successfully initialized in Read-Only Query Mode!")
print(f"   Model: {QUERY_LLM_MODEL} | Embedding: {EMBEDDING_MODEL} (1024 dims)")

---
## Step 4: Search Mode Comparison Benchmark 🔬
*(Demonstrates the empirical value of the Knowledge Graph for your thesis evaluation)*

LightRAG supports **5 distinct retrieval modes**:
1. **`naive`**: Traditional vector search over raw text chunks (**Baseline RAG**).
2. **`local`**: Focuses on fine-grained entity neighborhoods (Entity-level diagnostics).
3. **`global`**: Focuses on high-level relationship themes across the entire knowledge graph.
4. **`hybrid`**: Combines local entity graph traversal + global semantic search (**Flagship Mode**).
5. **`mix`**: Retrieves both structured graph entities/relations AND raw clinical text excerpts.

In [ ]:
import time

async def run_diagnostic_query(query: str, mode: str = "hybrid") -> str:
    """Execute query and display response with latency telemetry."""
    print(f"🔎 Query: '{query}'")
    print(f"⚙️ Search Mode: [{mode.upper()}]\n")
    
    start_t = time.time()
    response = await rag.aquery(query, param=QueryParam(mode=mode))
    elapsed = time.time() - start_t
    
    print("=" * 70)
    print(f"🏥 Clinical Response (Mode: {mode.upper()} | Time: {elapsed:.2f}s)")
    print("=" * 70)
    print(response)
    print("=" * 70 + "\n")
    return response

# Test on a classic differential diagnosis query:
benchmark_query = (
    "What are the characteristic clinical symptoms, laboratory markers, and imaging findings "
    "of Melioidosis (Burkholderia pseudomallei) compared to pulmonary Tuberculosis?"
)

# Test Hybrid Mode (Flagship)
hybrid_resp = await run_diagnostic_query(benchmark_query, mode="hybrid")

---
## Step 5: Clinical Diagnostic Benchmark Suite 🩺
Four realistic medical queries specifically designed for the **Tropical Diseases Dataset**:

- **Scenario 1:** Complex Differential Diagnosis (Prolonged Fever & Hepatosplenomegaly)
- **Scenario 2:** Radiologic Findings & Diagnostic Modalities (Pulmonary TB vs Pneumonia)
- **Scenario 3:** Pharmacotherapy & Treatment Monitoring (MDR-TB & Severe Malaria)
- **Scenario 4:** Severe Disease Progression & Warning Signs (Dengue Shock Syndrome)

In [ ]:
clinical_cases = [
    {
        "title": "Scenario 1: Complex Differential Diagnosis",
        "query": (
            "A 28-year-old patient from Southeast Asia presents with 3 weeks of high remittent fever, "
            "marked hepatosplenomegaly, and pancytopenia. What tropical diseases should be considered, "
            "and what specific laboratory or microscopic tests confirm each condition?"
        ),
        "mode": "hybrid"
    },
    {
        "title": "Scenario 2: Radiologic & Imaging Distinctions",
        "query": (
            "What specific imaging patterns on chest computed tomography (CT) distinguish "
            "active pulmonary tuberculosis from opportunistic fungal infections or bacterial pneumonia?"
        ),
        "mode": "mix"
    },
    {
        "title": "Scenario 3: Pharmacotherapy & Toxicities",
        "query": (
            "What are the standard first-line treatment regimens and crucial organ toxicity monitoring "
            "parameters for severe falciparum malaria and multi-drug resistant tuberculosis?"
        ),
        "mode": "local"
    },
    {
        "title": "Scenario 4: Severe Disease Progression Signs",
        "query": (
            "What critical clinical and hematological warning signs indicate progression from "
            "uncomplicated dengue fever to severe dengue shock syndrome (DSS)?"
        ),
        "mode": "hybrid"
    }
]

# Run all 4 clinical benchmark scenarios
for idx, item in enumerate(clinical_cases, 1):
    print(f"\n{'#' * 70}")
    print(f"Case {idx}: {item['title']}")
    print(f"{'#' * 70}")
    await run_diagnostic_query(item["query"], mode=item["mode"])

---
## Step 6: Visual Graph Explorer & Clinical Reasoning Paths 🕸️
Inspects the actual structure of the Knowledge Graph:
1. **Ego-Network Subgraph:** Extracts and visualizes all entities connected to a central medical topic (e.g., *Tuberculosis* or *Dengue*).
2. **Shortest Reasoning Path:** Traces how two seemingly separate medical concepts are bridged through clinical evidence (e.g., *Symptom* $\rightarrow$ *Test* $\rightarrow$ *Pathogen* $\rightarrow$ *Drug*).

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def visualize_clinical_subgraph(G: nx.Graph, central_entity: str, max_neighbors: int = 15):
    """Visualize the 1-hop clinical neighborhood around a key disease or symptom."""
    if central_entity not in G:
        matches = [n for n in G.nodes if central_entity.lower() in n.lower()]
        if matches:
            print(f"Exact match '{central_entity}' not found. Using closest match: '{matches[0]}'")
            central_entity = matches[0]
        else:
            print(f"Entity '{central_entity}' not found in the Knowledge Graph.")
            return

    # Extract neighbors
    neighbors = list(G.neighbors(central_entity))[:max_neighbors]
    sub_nodes = [central_entity] + neighbors
    subgraph = G.subgraph(sub_nodes)

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(subgraph, seed=42)

    # Color nodes: red for center, skyblue for neighbors
    node_colors = ['#e74c3c' if n == central_entity else '#3498db' for n in subgraph.nodes]

    nx.draw_networkx_nodes(subgraph, pos, node_color=node_colors, node_size=1600, alpha=0.9)
    nx.draw_networkx_edges(subgraph, pos, edge_color='#7f8c8d', width=1.8, alpha=0.6)
    nx.draw_networkx_labels(subgraph, pos, font_size=9, font_family='sans-serif', font_weight='bold')

    plt.title(f"Clinical Subgraph: 1-Hop Neighborhood of '{central_entity}'", fontsize=14, fontweight='bold', pad=15)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def find_clinical_reasoning_path(G: nx.Graph, source_entity: str, target_entity: str):
    """Find the shortest chain of clinical associations between two entities."""
    def match_node(query):
        if query in G: return query
        matches = [n for n in G.nodes if query.lower() in n.lower()]
        return matches[0] if matches else None

    src = match_node(source_entity)
    tgt = match_node(target_entity)

    if not src or not tgt:
        print(f"Could not resolve one of: '{source_entity}' ({src}) or '{target_entity}' ({tgt})")
        return

    try:
        path = nx.shortest_path(G, source=src, target=tgt)
        print(f"\n🔗 Clinical Reasoning Path between '{src}' and '{tgt}':")
        print("   " + "  -->  ".join([f"[{step}]" for step in path]))
        print(f"   (Path length: {len(path) - 1} diagnostic hops)")
    except nx.NetworkXNoPath:
        print(f"No direct path found between '{src}' and '{tgt}' in this graph component.")

# 1. Visualize ego-network around Tuberculosis (or Dengue)
visualize_clinical_subgraph(G, "Tuberculosis", max_neighbors=12)

# 2. Trace reasoning path from Symptom -> Drug
find_clinical_reasoning_path(G, "Fever", "Ceftriaxone")
find_clinical_reasoning_path(G, "Hemoptysis", "Isoniazid")

---
## Step 7: Interactive Diagnostic Console 💬
Type any custom clinical query below and press **Run** to query your Knowledge Graph!

**Available Modes:**
- `"hybrid"` — Best overall (graph traversal + vector retrieval)
- `"mix"` — Integrates graph nodes + raw case citations
- `"local"` — Fine-grained entity search
- `"global"` — Broad thematic patterns
- `"naive"` — Pure vector baseline (no graph)

In [ ]:
# Edit your question and preferred search mode here:
my_question = (
    "What are the common clinical presentations and diagnostic tests for Scrub Typhus (Orientia tsutsugamushi)?"
)
search_mode = "hybrid"  # "hybrid", "mix", "local", "global", or "naive"

# Run query
response = await run_diagnostic_query(my_question, mode=search_mode)